<a href="https://colab.research.google.com/github/astroelaa/miniragproject_/blob/main/mini_rag_project.ipynb%20" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import**

In [ ]:
!pip install -q langchain-text-splitters sentence-transformers transformers chromadb pypdf

In [ ]:
import os
import re
import shutil
import chromadb
from google.colab import files
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, os.path.join(DATA_DIR, filename))
os.listdir(DATA_DIR)

In [ ]:
reader = PdfReader("data/Savov_Notes.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() or ""

text = re.sub(r'[^\x00-\x7F]+', ' ', text)
text = re.sub(r'\s+', ' ', text)

print(len(text))
print(text[:300])

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = splitter.split_text(text)
sources = ["Savov_Notes.pdf"] * len(chunks)
len(chunks)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = model.encode(chunks, show_progress_bar=True)
chunk_embeddings.shape

In [ ]:
client = chromadb.EphemeralClient()

try:
    client.delete_collection("notes")
except Exception:
    pass

collection = client.create_collection("notes", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=[{"source": s} for s in sources],
)
collection.count()

store in Chroma

In [ ]:
client = chromadb.EphemeralClient()

try:
    client.delete_collection("notes")
except Exception:
    pass

collection = client.create_collection("notes", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=[{"source": s} for s in sources],
)
collection.count()

retrieval

In [ ]:
def retrieve(query, k=3, max_distance=0.8):
    query_vector = model.encode(query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=k)
    retrieved = []
    for text_chunk, distance, meta in zip(
        results["documents"][0], results["distances"][0], results["metadatas"][0]
    ):
        if distance <= max_distance:
            retrieved.append({"text": text_chunk, "distance": distance, "source": meta["source"]})
    return retrieved

context & prompt

In [ ]:
def build_context(retrieved):
    return "\n\n---\n\n".join(r["text"] for r in retrieved)

PROMPT_TEMPLATE = """Use only the context below to answer in a complete sentence. Do not use any outside knowledge.
If the context does not contain the answer, respond exactly: "I don't have enough information in the provided documents."

Context:
{context}

Question:
{question}

Answer:"""

def build_prompt(question, context):
    return PROMPT_TEMPLATE.format(context=context, question=question)

generation model

In [ ]:
gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

MAX_INPUT_TOKENS = 512

def generate_answer(question, context):
    overhead = PROMPT_TEMPLATE.format(context="", question=question)
    overhead_tokens = len(gen_tokenizer(overhead)["input_ids"])
    context_budget = MAX_INPUT_TOKENS - overhead_tokens

    context_ids = gen_tokenizer(context, truncation=True, max_length=context_budget)["input_ids"]
    context = gen_tokenizer.decode(context_ids, skip_special_tokens=True)

    prompt = build_prompt(question, context)
    inputs = gen_tokenizer(prompt, return_tensors="pt")
    output = gen_model.generate(**inputs, max_new_tokens=200)
    return gen_tokenizer.decode(output[0], skip_special_tokens=True)

def rag_pipeline(question, k=4):
    retrieved = retrieve(question, k=k)
    if not retrieved:
        return "I don't have enough information in the provided documents."
    context = build_context(retrieved)
    return generate_answer(question, context)

In [ ]:
questions = [
    "what is a diagonalizable matrix",
    "what does it mean for a matrix to be invertible",
    "what is the rank of a matrix",
    "what is an eigenvector",
]

for q in questions:
    print(q)
    for r in retrieve(q, k=4):
        print(round(r["distance"], 3), r["text"][:80])
    print(rag_pipeline(q))
    print()

In [ ]:
off_topic_questions = [
    "what's the weather in Cairo today",
    "who won the world cup in 2018",
]

for q in off_topic_questions:
    print(q)
    for r in retrieve(q, k=4):
        print(round(r["distance"], 3), r["text"][:80])
    print(rag_pipeline(q))
    print()